# Financial News Prompt Chaining

This notebook implements only the required prompt-chaining workflow:

`Ingest News -> Preprocess -> Classify -> Extract -> Summarize`

It is designed to run without paid APIs by using a small built-in financial news sample. If you have the Kaggle financial news CSV or a NewsAPI key, you can plug them into the ingestion cell.

## 1. Setup

The chain is intentionally modular: each stage receives structured output from the previous stage and returns structured output to the next stage.

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from html import unescape
import csv
import json
import os
import re
import textwrap
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import urlopen, Request


@dataclass
class NewsArticle:
    title: str
    description: str
    source: str = "unknown"
    published_at: str = ""
    url: str = ""


@dataclass
class ProcessedArticle:
    id: int
    title: str
    text: str
    source: str
    published_at: str
    url: str


SAMPLE_NEWS = [
    NewsArticle(
        title="Nvidia shares rise as analysts lift AI chip revenue forecasts",
        description="Several Wall Street analysts raised price targets for Nvidia after stronger demand checks for data center GPUs.",
        source="Sample Market Wire",
        published_at="2026-09-24T13:10:00Z",
    ),
    NewsArticle(
        title="Apple faces pressure after supplier report points to slower iPhone orders",
        description="A supplier note suggested softer near-term production schedules, weighing on Apple and related hardware names.",
        source="Sample Finance Daily",
        published_at="2026-09-24T15:40:00Z",
    ),
    NewsArticle(
        title="JPMorgan earnings beat estimates as net interest income remains resilient",
        description="The bank reported better-than-expected quarterly earnings, helped by credit quality and stable deposit trends.",
        source="Sample Earnings Desk",
        published_at="2026-09-24T20:05:00Z",
    ),
    NewsArticle(
        title="Fed officials signal caution on rate cuts as inflation progress slows",
        description="Bond yields moved higher after policymakers emphasized that inflation remains above target.",
        source="Sample Macro Brief",
        published_at="2026-09-24T21:15:00Z",
    ),
]

print(f"Notebook initialized at {datetime.now(timezone.utc).isoformat()} with {len(SAMPLE_NEWS)} sample articles.")

## 2. Prompt Chain Stage 1: Ingest News

This stage gathers raw financial news from one of three sources:

1. Kaggle-style CSV file, if `KAGGLE_FINANCIAL_NEWS_CSV` points to a local CSV.
2. NewsAPI.org, if `NEWSAPI_KEY` is available.
3. Built-in sample articles, so the notebook remains reproducible.

In [ ]:
def ingest_from_csv(path, limit=25):
    articles = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            title = row.get("title") or row.get("headline") or row.get("Title") or ""
            description = row.get("description") or row.get("text") or row.get("content") or row.get("Text") or ""
            if title or description:
                articles.append(NewsArticle(
                    title=title,
                    description=description,
                    source=row.get("source") or row.get("Source") or "kaggle_csv",
                    published_at=row.get("publishedAt") or row.get("date") or row.get("Date") or "",
                    url=row.get("url") or row.get("URL") or "",
                ))
            if len(articles) >= limit:
                break
    return articles


def ingest_from_newsapi(query="stock market OR earnings OR Federal Reserve", limit=10):
    api_key = os.getenv("NEWSAPI_KEY")
    if not api_key:
        return []
    params = urlencode({
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": limit,
        "apiKey": api_key,
    })
    request = Request(f"https://newsapi.org/v2/everything?{params}", headers={"User-Agent": "financial-prompt-chain-notebook"})
    with urlopen(request, timeout=20) as response:
        payload = json.loads(response.read().decode("utf-8"))
    articles = []
    for item in payload.get("articles", []):
        articles.append(NewsArticle(
            title=item.get("title") or "",
            description=item.get("description") or item.get("content") or "",
            source=(item.get("source") or {}).get("name", "newsapi"),
            published_at=item.get("publishedAt") or "",
            url=item.get("url") or "",
        ))
    return articles


def ingest_news(limit=25):
    csv_path = os.getenv("KAGGLE_FINANCIAL_NEWS_CSV")
    if csv_path and os.path.exists(csv_path):
        return ingest_from_csv(csv_path, limit=limit)
    newsapi_articles = ingest_from_newsapi(limit=limit)
    if newsapi_articles:
        return newsapi_articles
    return SAMPLE_NEWS[:limit]


raw_articles = ingest_news(limit=25)
print(f"Ingested {len(raw_articles)} articles")
for article in raw_articles[:3]:
    print("-", article.title)

## 3. Prompt Chain Stage 2: Preprocess

This stage cleans article text, removes noise, deduplicates items, and creates normalized records for downstream analysis.

In [ ]:
def clean_text(value):
    value = unescape(value or "")
    value = re.sub(r"https?://\S+", " ", value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def preprocess_articles(articles):
    processed = []
    seen = set()
    for index, article in enumerate(articles, start=1):
        title = clean_text(article.title)
        description = clean_text(article.description)
        combined = f"{title}. {description}".strip()
        fingerprint = re.sub(r"[^a-z0-9]+", "", combined.lower())[:180]
        if not combined or fingerprint in seen:
            continue
        seen.add(fingerprint)
        processed.append(ProcessedArticle(
            id=len(processed) + 1,
            title=title,
            text=combined,
            source=article.source,
            published_at=article.published_at,
            url=article.url,
        ))
    return processed


processed_articles = preprocess_articles(raw_articles)
print(f"Preprocessed {len(processed_articles)} unique articles")
processed_articles[0]

## 4. Prompt Chain Stage 3: Classify

This stage routes each article into a financial content class. In a production agentic system, this classifier could be an LLM prompt. Here, a transparent keyword classifier makes the notebook runnable offline.

In [ ]:
CLASS_KEYWORDS = {
    "earnings": ["earnings", "revenue", "profit", "guidance", "quarter", "estimates", "beat", "miss"],
    "macro": ["fed", "federal reserve", "inflation", "rates", "yield", "gdp", "jobs", "treasury"],
    "market_movement": ["shares", "stock", "rally", "rise", "fall", "selloff", "price target", "analysts"],
    "company_news": ["supplier", "demand", "orders", "product", "merger", "acquisition", "lawsuit"],
}


def classify_article(article):
    text = article.text.lower()
    scores = {
        label: sum(1 for keyword in keywords if keyword in text)
        for label, keywords in CLASS_KEYWORDS.items()
    }
    label, score = max(scores.items(), key=lambda item: item[1])
    if score == 0:
        label = "general_financial_news"
    confidence = min(0.95, 0.45 + (score * 0.15)) if score else 0.35
    return {"label": label, "confidence": round(confidence, 2), "scores": scores}


classified_articles = []
for article in processed_articles:
    classified_articles.append({
        "article": asdict(article),
        "classification": classify_article(article),
    })

for item in classified_articles:
    print(f"[{item['classification']['label']}] {item['article']['title']}")

## 5. Prompt Chain Stage 4: Extract

This stage extracts investment-relevant signals: tickers, companies, sentiment, catalysts, and risk terms.

In [ ]:
COMPANY_TO_TICKER = {
    "apple": "AAPL",
    "nvidia": "NVDA",
    "microsoft": "MSFT",
    "amazon": "AMZN",
    "tesla": "TSLA",
    "jpmorgan": "JPM",
    "meta": "META",
    "alphabet": "GOOGL",
}

POSITIVE_TERMS = {"rise", "raised", "stronger", "beat", "resilient", "better", "stable", "growth", "rally"}
NEGATIVE_TERMS = {"pressure", "slower", "softer", "weighing", "miss", "risk", "fall", "selloff", "inflation"}
CATALYST_TERMS = {"earnings", "guidance", "price targets", "demand", "orders", "rates", "inflation", "analysts"}
RISK_TERMS = {"inflation", "slower", "pressure", "softer", "rates", "yield", "lawsuit", "miss"}


def extract_signals(item):
    article = item["article"]
    text = article["text"]
    lower_text = text.lower()
    tickers = sorted({ticker for company, ticker in COMPANY_TO_TICKER.items() if company in lower_text})
    explicit_tickers = re.findall(r"\b[A-Z]{2,5}\b", text)
    tickers = sorted(set(tickers + [ticker for ticker in explicit_tickers if ticker not in {"AI", "GDP", "CEO", "CFO"}]))

    words = re.findall(r"[a-zA-Z]+", lower_text)
    positive_hits = [word for word in words if word in POSITIVE_TERMS]
    negative_hits = [word for word in words if word in NEGATIVE_TERMS]
    sentiment_score = len(positive_hits) - len(negative_hits)
    sentiment = "positive" if sentiment_score > 0 else "negative" if sentiment_score < 0 else "neutral"

    catalysts = sorted(term for term in CATALYST_TERMS if term in lower_text)
    risks = sorted(term for term in RISK_TERMS if term in lower_text)

    return {
        "article_id": article["id"],
        "title": article["title"],
        "class": item["classification"]["label"],
        "class_confidence": item["classification"]["confidence"],
        "tickers": tickers,
        "sentiment": sentiment,
        "sentiment_score": sentiment_score,
        "positive_terms": positive_hits,
        "negative_terms": negative_hits,
        "catalysts": catalysts,
        "risks": risks,
        "source": article["source"],
    }


extracted_signals = [extract_signals(item) for item in classified_articles]
print(json.dumps(extracted_signals, indent=2))

## 6. Prompt Chain Stage 5: Summarize

The final stage turns extracted signals into a concise investment-research brief.

In [ ]:
def summarize_signals(signals):
    by_class = Counter(signal["class"] for signal in signals)
    by_sentiment = Counter(signal["sentiment"] for signal in signals)
    ticker_counts = Counter(ticker for signal in signals for ticker in signal["tickers"])
    catalysts = Counter(catalyst for signal in signals for catalyst in signal["catalysts"])
    risks = Counter(risk for signal in signals for risk in signal["risks"])

    top_items = sorted(signals, key=lambda signal: (abs(signal["sentiment_score"]), signal["class_confidence"]), reverse=True)

    brief = []
    brief.append("Financial News Prompt-Chain Brief")
    brief.append("=" * 36)
    brief.append(f"Articles analyzed: {len(signals)}")
    brief.append(f"Content mix: {dict(by_class)}")
    brief.append(f"Sentiment mix: {dict(by_sentiment)}")
    brief.append(f"Most mentioned tickers: {dict(ticker_counts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading catalysts: {dict(catalysts.most_common(5)) or 'none detected'}")
    brief.append(f"Leading risks: {dict(risks.most_common(5)) or 'none detected'}")
    brief.append("")
    brief.append("Key article-level takeaways:")
    for signal in top_items:
        ticker_text = ", ".join(signal["tickers"]) if signal["tickers"] else "broad market"
        catalyst_text = ", ".join(signal["catalysts"]) if signal["catalysts"] else "no explicit catalyst"
        risk_text = ", ".join(signal["risks"]) if signal["risks"] else "no major risk term"
        brief.append(
            f"- {signal['title']} | class={signal['class']} | tickers={ticker_text} | "
            f"sentiment={signal['sentiment']} | catalysts={catalyst_text} | risks={risk_text}"
        )
    return "\n".join(brief)


final_summary = summarize_signals(extracted_signals)
print(final_summary)

## 7. End-to-End Chain Function

This wrapper makes the prompt-chaining pattern explicit and reusable.

In [ ]:
def run_prompt_chain(limit=25):
    raw = ingest_news(limit=limit)
    processed = preprocess_articles(raw)
    classified = [{"article": asdict(article), "classification": classify_article(article)} for article in processed]
    extracted = [extract_signals(item) for item in classified]
    summary = summarize_signals(extracted)
    return {
        "raw_count": len(raw),
        "processed_count": len(processed),
        "classified": classified,
        "extracted_signals": extracted,
        "summary": summary,
    }


chain_result = run_prompt_chain(limit=25)
print(chain_result["summary"])

## 8. Why Prompt Chaining Matters

This project uses prompt chaining to break financial analysis into smaller, reliable steps instead of asking one model or function to do everything at once. Each stage produces structured output that becomes the input for the next stage.

| Chain Step | Purpose | Output |
|---|---|---|
| Ingest News | Pull financial news from NewsAPI, Kaggle CSV, or sample data | Raw articles |
| Preprocess | Clean, normalize, and deduplicate article text | Clean article records |
| Classify | Identify the type of financial content | Earnings, macro, market movement, company news |
| Extract | Pull investment signals from each article | Tickers, sentiment, catalysts, risks |
| Summarize | Convert extracted signals into a research brief | Final financial news summary |

Separating the workflow this way makes the system easier to debug, evaluate, and improve. If one stage produces weak results, that stage can be refined without rewriting the entire pipeline.

## 9. Example Output

The notebook produces a financial news brief that includes:

- Number of articles analyzed
- Content mix by category
- Sentiment breakdown
- Most mentioned tickers
- Leading catalysts
- Leading risks
- Article-level investment takeaways

These outputs demonstrate that the chain does more than summarize text: it transforms raw financial news into structured investment signals and then converts those signals into a readable research brief.